# LangChain Agents — What Changed & What's New

## What happened to `AgentExecutor`?

`AgentExecutor` and `create_react_agent` moved to a separate package called **langchain-classic**. They still exist but are no longer actively maintained. The main `langchain` package dropped them starting from version 0.3+.

---

## Why it was removed

- It was a black box — hard to customize, debug, or extend
- It relied on text-based ReAct format (`Action: ... Action Input: ...`) which broke easily if the LLM didn't format output perfectly
- Modern LLMs (Gemini, GPT-4, Claude) now support **native tool calling via API** — no need to parse text anymore
- **LangGraph** was built as the proper replacement for production agents

---

## New Way 1 — Manual Tool Loop (Simple)

Best for: learning, simple single-agent tasks

```python
from langchain_core.messages import HumanMessage, ToolMessage

llm_with_tools = llm.bind_tools(tools)
tool_map = {t.name: t for t in tools}

messages = [HumanMessage("your query here")]

while True:
    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if not response.tool_calls:
        break

    for tc in response.tool_calls:
        result = tool_map[tc["name"]].invoke(tc["args"])
        messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

print(response.content)
```

What's happening:
- `bind_tools` tells the LLM what tools are available
- LLM returns structured `tool_calls` (not raw text)
- You execute the tool and feed result back
- Loop continues until LLM stops calling tools

---

## New Way 2 — LangGraph (Production)

Best for: multi-agent systems, complex workflows, memory, branching

```python
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(llm, tools)
response = agent.invoke({"messages": [HumanMessage("your query")]})
print(response["messages"][-1].content)
```

LangGraph gives you:
- Full control over agent state
- Built-in memory and checkpointing
- Multi-agent coordination
- Error handling and retries
- Human-in-the-loop support

---

## Quick Comparison

| Feature | Old (AgentExecutor) | New (Manual Loop) | New (LangGraph) |
|---|---|---|---|
| Setup | Complex | Simple | Medium |
| Customization | Low | Full | Full |
| Multi-agent | No | No | Yes |
| Production ready | No | Partial | Yes |
| Reliability | Low (text parsing) | High (native tool call) | High |

---

## TL;DR

> LLMs now call tools natively via API. No need to parse `Action: ...` text anymore. For simple tasks use the manual loop. For production use LangGraph.

In [21]:
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.6 MB/s eta 0:00:00


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="add api key here",  # free at console.groq.com
    temperature=0
)

llm_with_tools = llm.bind_tools(tools)

: 

In [23]:
from langchain_core.messages import HumanMessage, ToolMessage

messages = [HumanMessage("Find the capital of Madhya Pradesh, then find its current weather condition")]

while True:
    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if not response.tool_calls:
        break

    for tc in response.tool_calls:
        result = tool_map[tc["name"]].invoke(tc["args"])
        messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

print(response.content)

The current weather condition in Bhopal, the capital of Madhya Pradesh, is Mist with a temperature of 25°C, humidity of 100%, wind speed of 16.6 km/h, and a 17% chance of rain.
